# Reducción de Dimensionalidad con t-SNE
## Dataset: Hotel Bookings

**t-SNE** (t-distributed Stochastic Neighbor Embedding) proyecta datos de alta
dimensionalidad a 2D o 3D preservando la estructura local (vecindades entre puntos).
Es especialmente útil para revelar agrupaciones naturales en datos complejos.

### Diferencias con ACP
| | ACP | t-SNE |
|-|-----|-------|
| Tipo | Lineal | No lineal |
| Objetivo | Varianza máxima | Estructura local |
| Proyectar nuevos datos | Sí | No |
| Velocidad | Muy rápido | Lento en datasets grandes |

### Flujo de trabajo
```
mf.TSNE(path, num)          # 1. Cargar datos
    ↓ métodos EDA heredados  # 2. Limpiar y preparar
tsne.ajustar()              # 3. Calcular proyección
tsne.plot_proyeccion()      # 4. Visualizar
```

## 1. Importación de librerías

In [ ]:
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
import pckEDA as mf

%matplotlib inline
print('Clase TSNE disponible:', mf.TSNE)

## 2. Carga de datos

In [ ]:
tsne = mf.TSNE('../datasets/hotel_bookings.csv', 1, n_componentes=2, perplejidad=30)

tsne.mostrarTamaño()
tsne.muestraTiposDeDatos()

## 3. Preprocesamiento con métodos heredados del EDA

In [ ]:
tsne.codificarCategorica('hotel')
tsne.codificarCategorica('arrival_date_month')
tsne.codificarCategorica('assigned_room_type')
tsne.codificarCategorica('deposit_type', mapeo={'No Deposit': 0, 'Non Refund': 1, 'Refundable': 2})
tsne.codificarCategorica('customer_type')
tsne.codificarCategorica('reservation_status')
tsne.eliminarDuplicados()
tsne.eliminarNulos()
tsne.analisisNumerico()
print(f'Observaciones para la proyección: {len(tsne.df)}')

## 4. Ajuste del modelo t-SNE

In [ ]:
tsne.ajustar()

print(f'Perplejidad     : {tsne.perplejidad}')
print(f'Iteraciones     : {tsne.iteraciones}')
print(f'Dimensiones     : {tsne.n_componentes}')
print(f'Shape proyección: {tsne.coordenadas.shape}')

## 5. Proyección sin etiquetas

In [ ]:
plt.figure(figsize=(10, 8))
tsne.plot_proyeccion(titulo='t-SNE — Hotel Bookings (sin etiquetas)')
plt.tight_layout()
plt.show()

## 6. Proyección coloreada con clusters de K-Means

Combinamos t-SNE (visualización) con K-Means (clustering) para ver si los clusters
encontrados por K-Means tienen coherencia geométrica en el espacio t-SNE.

In [ ]:
# Reutilizamos los datos ya preprocesados del objeto tsne
import pandas as pd
km = mf.KMeans('../datasets/hotel_bookings.csv', 1, n_clusters=4)
km.codificarCategorica('hotel')
km.codificarCategorica('arrival_date_month')
km.codificarCategorica('assigned_room_type')
km.codificarCategorica('deposit_type', mapeo={'No Deposit': 0, 'Non Refund': 1, 'Refundable': 2})
km.codificarCategorica('customer_type')
km.codificarCategorica('reservation_status')
km.eliminarDuplicados()
km.eliminarNulos()
km.analisisNumerico()
km.ajustar()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

plt.sca(axes[0])
tsne.plot_proyeccion(titulo='t-SNE — sin etiquetas')

plt.sca(axes[1])
tsne.plot_proyeccion(etiquetas=km.etiquetas, titulo='t-SNE — coloreado por K-Means (k=4)')

plt.tight_layout()
plt.show()

## 7. Efecto de la perplejidad

In [ ]:
tsne.plot_perplejidad(valores=(5, 15, 30, 50), titulo='Efecto de la Perplejidad — Hotel Bookings')
plt.show()

## 8. Conclusiones

- t-SNE revela **agrupaciones no lineales** que ACP no puede capturar.
- La **perplejidad** controla el equilibrio entre estructura local y global — prueba varios valores antes de interpretar el resultado.
- Las distancias absolutas entre clusters en el gráfico t-SNE **no son directamente interpretables** — solo la proximidad relativa entre puntos tiene significado.
- Colorear la proyección con etiquetas de K-Means permite validar si los clusters tienen coherencia geométrica.